In [3]:
!pip install -q pandas==2.2.2 numpy==2.0.2
!pip install -q sentence-transformers==4.1.0 transformers==4.51.3 peft bitsandbytes accelerate faiss-cpu rank-bm25 rouge-score nltk scikit-learn

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [15]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [1]:
import os
import re
import json
import torch
import faiss
import numpy as np
import pandas as pd

from collections import Counter

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
DATASET_DIR = "/content/Datasets_Ceng493_legal_rag"

CORPUS_PATH = f"{DATASET_DIR}/corpus.jsonl"
GOLD_PATH = f"{DATASET_DIR}/gold_benchmark.json"

FT_RERANKER_PATH = "/content/drive/MyDrive/CENG493_models/finetuned_legal_reranker"
FT_LLM_ADAPTER_PATH = "/content/drive/MyDrive/qwen2_5_7b_legal_lora_adapter"

print("Dataset files:", os.listdir(DATASET_DIR))
print("Reranker exists:", os.path.exists(FT_RERANKER_PATH))
print("Reranker config:", os.path.exists(os.path.join(FT_RERANKER_PATH, "config.json")))
print("LLM adapter exists:", os.path.exists(FT_LLM_ADAPTER_PATH))

Dataset files: ['gold_benchmark.json', 'rag_eval.json', 'corpus.jsonl', 'embedding.jsonl', 'llm.jsonl', 'reranker.jsonl']
Reranker exists: True
Reranker config: True
LLM adapter exists: True


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def clean_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_tr(text):
    return normalize_text(text).split()

In [5]:
corpus = []

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            corpus.append(json.loads(line))

documents = []
metadatas = []

for item in corpus:
    text = clean_text(item.get("text", ""))

    if len(text) > 20:
        documents.append(text)
        metadatas.append({
            "doc_id": item.get("id"),
            "title": item.get("title", ""),
            "metadata": item.get("metadata", {})
        })

print("Corpus:", len(corpus))
print("Usable documents:", len(documents))
print("Sample:", documents[0][:300])

Corpus: 7579
Usable documents: 7579
Sample: Susma hakkı, kişinin kendi lehine veya aleyhine ifade verme kararını özgürce kullanabilmesine olanak tanır.


In [6]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

doc_embeddings = embedding_model.encode(
    documents,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

doc_embeddings = np.asarray(doc_embeddings).astype("float32")

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print("Embedding shape:", doc_embeddings.shape)
print("FAISS index:", index.ntotal)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Embedding shape: (7579, 384)
FAISS index: 7579


In [7]:
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

print("BM25 corpus:", len(tokenized_docs))

BM25 corpus: 7579


In [8]:
reranker_model = CrossEncoder(
    FT_RERANKER_PATH,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Fine-tuned legal reranker loaded from:", FT_RERANKER_PATH)

Fine-tuned legal reranker loaded from: /content/drive/MyDrive/CENG493_models/finetuned_legal_reranker


In [9]:
def dense_retrieve(query, k=100):
    q_emb = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(q_emb, k)

    results = []

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        results.append({
            "idx": int(idx),
            "rank": rank,
            "dense_score": float(score),
            "text": documents[idx],
            "meta": metadatas[idx],
            "method": "dense"
        })

    return results


def bm25_retrieve(query, k=100):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:k]

    results = []

    for rank, idx in enumerate(top_idx, start=1):
        results.append({
            "idx": int(idx),
            "rank": rank,
            "bm25_score": float(scores[idx]),
            "text": documents[idx],
            "meta": metadatas[idx],
            "method": "bm25"
        })

    return results


def hybrid_candidates(query, dense_k=100, bm25_k=100, dense_weight=0.5, bm25_weight=0.5):
    dense_docs = dense_retrieve(query, k=dense_k)
    bm25_docs = bm25_retrieve(query, k=bm25_k)

    candidates = {}

    for doc in dense_docs:
        idx = doc["idx"]
        candidates[idx] = doc.copy()
        candidates[idx]["fusion_score"] = candidates[idx].get("fusion_score", 0) + dense_weight * (1 / doc["rank"])

    for doc in bm25_docs:
        idx = doc["idx"]

        if idx not in candidates:
            candidates[idx] = doc.copy()
            candidates[idx]["fusion_score"] = 0

        candidates[idx]["fusion_score"] += bm25_weight * (1 / doc["rank"])

    return sorted(candidates.values(), key=lambda x: x["fusion_score"], reverse=True)


def finetuned_rerank(query, candidates, top_k=10, rerank_k=80):
    candidates = candidates[:rerank_k]

    pairs = [(query, cand["text"]) for cand in candidates]

    scores = reranker_model.predict(
        pairs,
        batch_size=32,
        show_progress_bar=False
    )

    reranked = []

    for cand, score in zip(candidates, scores):
        nd = cand.copy()
        nd["ft_reranker_score"] = float(score)
        reranked.append(nd)

    return sorted(reranked, key=lambda x: x["ft_reranker_score"], reverse=True)[:top_k]


def final_retrieve(query, top_k=10):
    candidates = hybrid_candidates(
        query,
        dense_k=100,
        bm25_k=100,
        dense_weight=0.5,
        bm25_weight=0.5
    )

    return finetuned_rerank(
        query,
        candidates,
        top_k=top_k,
        rerank_k=80
    )

In [10]:
def split_sentences_tr(text):
    text = clean_text(text)
    # Nokta, soru işareti, ünlem ve satır sonu benzeri ayrımlar
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    return sentences


def keyword_overlap(query, text):
    q_tokens = set(tokenize_tr(query))
    t_tokens = set(tokenize_tr(text))

    stopwords = {
        "nedir", "ne", "göre", "hangi", "bir", "ve", "ile", "mi", "mı", "mu", "mü",
        "de", "da", "bu", "şu", "o", "olarak", "için", "maddesi", "madde",
        "nasıl", "nelerdir", "kimdir", "türkiye", "hukuk", "kanun"
    }

    q_tokens = {t for t in q_tokens if t not in stopwords and len(t) > 1}

    if not q_tokens:
        return 0.0

    return len(q_tokens.intersection(t_tokens)) / len(q_tokens)


def extract_article_no(question):
    m = re.search(r"\bmadde\s+(\d+)\b", normalize_text(question))
    return m.group(1) if m else None


def extract_query_phrases(question):
    q = normalize_text(question)
    tokens = [t for t in tokenize_tr(question) if len(t) > 2]

    stopwords = {
        "nedir", "göre", "hangi", "nasıl", "nelerdir",
        "madde", "maddesi", "türkiye", "hukuk", "kanun"
    }

    tokens = [t for t in tokens if t not in stopwords]

    phrases = []

    # 2-gram ve 3-gram phrase çıkar
    for n in [3, 2]:
        for i in range(len(tokens) - n + 1):
            phrase = " ".join(tokens[i:i+n])
            if len(phrase) > 5:
                phrases.append(phrase)

    return phrases


def phrase_match_score(question, sentence):
    s = normalize_text(sentence)
    phrases = extract_query_phrases(question)

    if not phrases:
        return 0.0

    score = 0.0

    for phrase in phrases:
        if phrase in s:
            score += 2.5

    # Eğer önemli phrase hiç geçmiyorsa hafif ceza
    if score == 0 and len(phrases) > 0:
        score -= 1.5

    return score


def sentence_score(question, sentence, doc_title=""):
    q = normalize_text(question)
    s = normalize_text(sentence)

    score = 0.0

    # Genel lexical overlap
    score += 2.0 * keyword_overlap(question, sentence)
    score += 1.0 * keyword_overlap(question, doc_title)

    # Çok kelimeli hukuk terimi eşleşmesi
    score += phrase_match_score(question, sentence)

    # Madde numarası eşleşmesi
    article_no = extract_article_no(question)
    if article_no:
        if re.search(rf"\bmadde\s+{article_no}\b", s):
            score += 5.0
        else:
            score -= 1.0

    # Tanım sorularında tanım marker bonusu
    if "nedir" in q or "ne demektir" in q:
        definition_markers = [
            "ifade eder",
            "tanımlanır",
            "denir",
            "anlamına gelir",
            "olarak adlandırılır",
            "türüdür",
            "hakkıdır",
            "davasıdır",
            "tazminattır",
            "paradır",
            "hesaplanır",
            "ödenir"
        ]

        if any(marker in s for marker in definition_markers):
            score += 2.0

    # Boilerplate cezası
    boilerplate = [
        "daha fazla bilgi",
        "başlıklı yazı",
        "inceleyebilir",
        "ulaşılabilir",
        "edinilebilir",
        "arama motoru"
    ]

    if any(p in s for p in boilerplate):
        score -= 3.0

    # Aşırı kısa / aşırı uzun cezası
    n_words = len(sentence.split())

    if n_words < 4:
        score -= 2.0

    if n_words > 90:
        score -= 1.0

    return score


def select_answer_context(question, retrieved_docs, max_sentences=5):
    candidates = []

    for doc_rank, doc in enumerate(retrieved_docs[:10], start=1):
        title = doc["meta"].get("title", "")
        doc_id = doc["meta"].get("doc_id", "")

        sentences = split_sentences_tr(doc["text"])

        if not sentences:
            sentences = [doc["text"][:500]]

        for sent in sentences:
            candidates.append({
                "sentence": sent,
                "doc_rank": doc_rank,
                "doc_id": doc_id,
                "title": title,
                "score": sentence_score(question, sent, title)
            })

    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)

    selected = []
    used_texts = set()

    for c in candidates:
        norm = normalize_text(c["sentence"])

        if norm in used_texts:
            continue

        used_texts.add(norm)
        selected.append(c)

        if len(selected) >= max_sentences:
            break

    return selected

In [11]:
q = "Kıdem tazminatı nedir?"

retrieved = final_retrieve(q, top_k=10)
selected_context = select_answer_context(q, retrieved, max_sentences=5)

print("QUESTION:", q)

for i, c in enumerate(selected_context, start=1):
    print("\nSnippet", i)
    print("score:", c["score"])
    print("doc_rank:", c["doc_rank"])
    print("doc_id:", c["doc_id"])
    print(c["sentence"])

QUESTION: Kıdem tazminatı nedir?

Snippet 1
score: 6.5
doc_rank: 2
doc_id: lawchatbot_turk_borclar_kanunu_borclar_hukuku_is_hukuku_0006
kıdem tazminatı, işçinin her bir yıllık çalışması karşılığında 30 günlük brüt ücreti üzerinden hesaplanır ve iş akdinin sona ermesiyle ödenir.

Snippet 2
score: 6.5
doc_rank: 5
doc_id: oricon_is_sgk_borclar_000023
Kıdem tazminatı, işçinin fesih tarihine kadarki hizmetlerine karşılık işveren tarafından ödenmesi gereken toplu paradır ve işçi, işverene ait işyerinde en az bir yıl çalışmışsa kıdem tazminatı talep edebilir.

Snippet 3
score: 4.5
doc_rank: 2
doc_id: lawchatbot_turk_borclar_kanunu_borclar_hukuku_is_hukuku_0006
bu durumda işçi, kıdem tazminatı talep edebilir.

Snippet 4
score: 4.5
doc_rank: 2
doc_id: lawchatbot_turk_borclar_kanunu_borclar_hukuku_is_hukuku_0006
kıdem tazminatı, işçinin çalışma süresi ve ücreti göz önünde bulundurularak belirlenir.

Snippet 5
score: 4.5
doc_rank: 2
doc_id: lawchatbot_turk_borclar_kanunu_borclar_hukuku_is_hukuku_

In [12]:
def build_grounded_prompt(question, selected_context):
    context = "\n\n".join([
        f"[Kaynak {i+1} | doc_id={c['doc_id']} | title={c['title']}]\n{c['sentence']}"
        for i, c in enumerate(selected_context)
    ])

    prompt = f"""
Aşağıdaki Türkçe hukuk sorusunu yalnızca verilen kaynak cümlelerine göre cevapla.

Kurallar:
- Kaynak cümlelerinde geçmeyen hiçbir bilgi ekleme.
- Yeni tarih, sayı, oran, madde veya yorum üretme.
- Cevap 1 kısa cümle olsun.
- Cevap mümkün olduğunca kaynak cümlesindeki ifadeye yakın olsun.
- Cevabın sonunda kullandığın kaynak numarasını yaz. Örnek: [Kaynak 1]
- Eğer kaynak cümlelerinde cevap yoksa yalnızca şunu yaz: Verilen kaynaklarda bu sorunun cevabı bulunmamaktadır.

Soru:
{question}

Kaynak cümleleri:
{context}

Cevap:
"""
    return prompt

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    base_model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa"
)

base_model.eval()

print("Base 7B BF16 loaded.")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Base 7B BF16 loaded.


In [14]:
ft_base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa"
)

ft_model = PeftModel.from_pretrained(
    ft_base_model,
    FT_LLM_ADAPTER_PATH,
    is_trainable=False
)

ft_model.eval()

print("Fine-tuned 7B BF16 loaded.")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Fine-tuned 7B BF16 loaded.


In [15]:
def generate_grounded_answer(model, question, selected_context, max_new_tokens=70):
    prompt = build_grounded_prompt(question, selected_context)

    messages = [
        {
            "role": "system",
            "content": (
                "Sen Türk hukuk alanında çalışan bir RAG asistanısın. "
                "Yalnızca verilen kaynak cümlelerine bağlı kal. "
                "Kaynakta olmayan bilgi üretme. "
                "Cevabı Türkçe, sade ve tek cümle ver."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    gen_config = model.generation_config
    gen_config.do_sample = False
    gen_config.temperature = None
    gen_config.top_p = None
    gen_config.top_k = None

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            generation_config=gen_config,
            max_new_tokens=max_new_tokens,
            repetition_penalty=1.03,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    answer = re.sub(r"\s+", " ", answer).strip()

    return answer

In [16]:
def looks_broken_turkish(answer):
    bad_patterns = [
        "kaynar", "kaynan", "cumhuir", "cumhh", "таз", "з", "修", "요청",
        "devletي", "ร"
    ]

    a = str(answer).lower()

    if any(p in a for p in bad_patterns):
        return True

    # Çok fazla garip karakter varsa
    weird_chars = re.findall(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ0-9\s.,;:()\[\]'-]", str(answer))
    if len(weird_chars) >= 3:
        return True

    return False


def source_overlap_simple(answer, selected_context):
    answer_tokens = set(tokenize_tr(answer))

    stopwords = {
        "ve", "veya", "ile", "bir", "bu", "şu", "o", "da", "de",
        "için", "olarak", "göre", "kaynak", "cevap"
    }

    answer_tokens = {t for t in answer_tokens if t not in stopwords and len(t) > 1}

    if not answer_tokens:
        return 0.0

    best = 0.0

    for c in selected_context:
        source_tokens = set(tokenize_tr(c["sentence"]))
        overlap = len(answer_tokens.intersection(source_tokens)) / len(answer_tokens)
        best = max(best, overlap)

    return best


def grounded_fallback_answer(selected_context):
    if not selected_context:
        return "Verilen kaynaklarda bu sorunun cevabı bulunmamaktadır."

    best = selected_context[0]
    return f"{best['sentence']} [Kaynak 1]"


def generate_verified_answer(model, question, selected_context):
    answer = generate_grounded_answer(
        model,
        question,
        selected_context
    )

    overlap = source_overlap_simple(answer, selected_context)

    if looks_broken_turkish(answer) or overlap < 0.35:
        answer = grounded_fallback_answer(selected_context)

    return answer

In [17]:
def ask_legal_rag(question, model=base_model, max_sentences=5):
    retrieved_docs = final_retrieve(question, top_k=10)
    selected_context = select_answer_context(
        question,
        retrieved_docs,
        max_sentences=max_sentences
    )

    answer = generate_verified_answer(
    model,
    question,
    selected_context
)

    print("=" * 100)
    print("SORU:")
    print(question)

    print("\nCEVAP:")
    print(answer)

    print("\nKULLANILAN KAYNAK CÜMLELERİ:")
    for i, c in enumerate(selected_context, start=1):
        print(f"\n[Kaynak {i}]")
        print("doc_id:", c["doc_id"])
        print("title:", c["title"])
        print("score:", c["score"])
        print(c["sentence"])

    return {
        "question": question,
        "answer": answer,
        "selected_context": selected_context,
        "retrieved_docs": retrieved_docs
    }

In [18]:
test_questions = [
    "Anayasa madde 1'e göre Türkiye'nin devlet şekli nedir?",
    "Kıdem tazminatı nedir?",
    "İdari dava nedir?",
    "Hak arama hürriyeti nedir?"
]

for q in test_questions:
    ask_legal_rag(q, model=base_model)

SORU:
Anayasa madde 1'e göre Türkiye'nin devlet şekli nedir?

CEVAP:
Madde 1 – Türkiye Devleti bir Cumhuriyettir. [Kaynak 1]

KULLANILAN KAYNAK CÜMLELERİ:

[Kaynak 1]
doc_id: turkish_law_eski_2709_turkiye_cumhuriyeti_anayasasi_m1
title: Türkiye Cumhuriyeti Anayasası
score: 3.5
Madde 1 – Türkiye Devleti bir Cumhuriyettir.

[Kaynak 2]
doc_id: lawchatbot_turkiye_cumhuriyeti_anayasasi_turk_borclar_kanunu_borc_0006
title: Türkiye Cumhuriyeti Anayasası + Türk Borçlar Kanunu / Borçlar Hukuku
score: 0.5
Devletin şekli, Türkiye Cumhuriyeti'nin demokratik, laik ve sosyal bir hukuk devleti olduğunu ifade eder.

[Kaynak 3]
doc_id: lawchatbot_turkiye_cumhuriyeti_anayasasi_turk_borclar_kanunu_borc_0006
title: Türkiye Cumhuriyeti Anayasası + Türk Borçlar Kanunu / Borçlar Hukuku
score: 0.5
Anayasa'nın üstünlüğü, tüm yasaların ve devlet organlarının Anayasa'ya uygun olması gerektiğini ifade eder.

[Kaynak 4]
doc_id: lawchatbot_turkiye_cumhuriyeti_anayasasi_turk_borclar_kanunu_borc_0006
title: Türkiye C

In [19]:
rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=False
)

smooth = SmoothingFunction().method1


def normalize_answer(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))


def token_f1(pred, gold):
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


def bleu_score(pred, gold):
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    return sentence_bleu(
        [gold_tokens],
        pred_tokens,
        smoothing_function=smooth
    )


def rouge_l(pred, gold):
    return rouge.score(str(gold), str(pred))["rougeL"].fmeasure


def source_overlap_answer(answer, selected_context):
    answer_tokens = set(tokenize_tr(answer))

    stopwords = {
        "ve", "veya", "ile", "bir", "bu", "şu", "o", "da", "de",
        "için", "olarak", "göre", "kaynak", "cevap"
    }

    answer_tokens = {
        t for t in answer_tokens
        if t not in stopwords and len(t) > 1
    }

    if not answer_tokens:
        return 0.0

    best_overlap = 0.0

    for c in selected_context:
        source_tokens = set(tokenize_tr(c["sentence"]))
        overlap = len(answer_tokens.intersection(source_tokens)) / len(answer_tokens)
        best_overlap = max(best_overlap, overlap)

    return best_overlap


def faithfulness_score(answer, selected_context):
    return source_overlap_answer(answer, selected_context)


def citation_present(answer):
    return int("kaynak" in str(answer).lower())


def citation_accuracy(answer, selected_context):
    if not selected_context:
        return 0

    answer_str = str(answer)

    match = re.search(r"Kaynak\s*(\d+)", answer_str, flags=re.IGNORECASE)

    if match:
        source_idx = int(match.group(1)) - 1
        source_idx = max(0, min(source_idx, len(selected_context) - 1))
    else:
        source_idx = 0

    source_sentence = selected_context[source_idx]["sentence"]

    answer_tokens = set(tokenize_tr(answer))
    source_tokens = set(tokenize_tr(source_sentence))

    stopwords = {
        "ve", "veya", "ile", "bir", "bu", "şu", "o", "da", "de",
        "için", "olarak", "göre", "kaynak", "cevap"
    }

    answer_tokens = {
        t for t in answer_tokens
        if t not in stopwords and len(t) > 1
    }

    if not answer_tokens:
        return 0

    overlap = len(answer_tokens.intersection(source_tokens)) / len(answer_tokens)

    return int(overlap >= 0.35)


def hallucination_rate(answer, selected_context):
    return 1 - faithfulness_score(answer, selected_context)

In [26]:
gold_df = pd.read_json(GOLD_PATH)

eval_df_final = gold_df.sample(
    n=min(100, len(gold_df)),
    random_state=42
).reset_index(drop=True)

print("Gold size:", len(gold_df))
print("Eval size:", len(eval_df_final))
eval_df_final.head()

Gold size: 240
Eval size: 100


,question_id,question,verified_answer,gold_sources,answer_type,difficulty,benchmark_status,manual_legal_review_recommended,validation_flags,quality_profile,notes
0,gold_final_0025,Ceza Muhakemesi Kanunu m.80 kapsamında “Geneti...,Kaynağa göre: Madde 80 – (Değişik: 25/5/2005 –...,[{'source_id': 'turkish_law_eski_5271_ceza_muh...,extractive_source_grounded,medium,auto_source_verified_review_ready,True,[],"{'source_id_exists_in_corpus': True, 'citation...",Soru-cevap çifti corpus_final_strict_verified....
1,gold_final_0007,Ceza Muhakemesi Kanunu m.2 kapsamında “Tanımla...,Kaynağa göre: Madde 2 – (1) Bu Kanunun uygulan...,[{'source_id': 'turkish_law_eski_5271_ceza_muh...,extractive_source_grounded,hard,auto_source_verified_review_ready,True,[],"{'source_id_exists_in_corpus': True, 'citation...",Soru-cevap çifti corpus_final_strict_verified....
2,gold_final_0094,"Hukuk Genel Kurulu 2024/382 E., 2024/264 K. sa...",Karar sonucu göstergesi: Gönderme. Uyuşmazlık ...,"[{'source_id': 'yargitay_0066_is_hukuku_001', ...",extractive_source_grounded,easy,auto_source_verified_review_ready,True,[restored_retrieval_source_not_official_text],"{'source_id_exists_in_corpus': True, 'citation...",Soru-cevap çifti corpus_final_strict_verified....
3,gold_final_0110,"Hukuk Genel Kurulu 2013/1415 E., 2015/555 K. s...",Karar sonucu göstergesi: Bozma. Uyuşmazlık öze...,[{'source_id': 'yargitay_0596_tasinmaz_hukuku_...,extractive_source_grounded,medium,auto_source_verified_review_ready,True,[restored_retrieval_source_not_official_text],"{'source_id_exists_in_corpus': True, 'citation...",Soru-cevap çifti corpus_final_strict_verified....
4,gold_final_0105,"Hukuk Genel Kurulu 2024/560 E., 2024/407 K. sa...",Karar sonucu göstergesi: Gönderme. Uyuşmazlık ...,[{'source_id': 'yargitay_0124_medeni_hukuk_001...,extractive_source_grounded,easy,auto_source_verified_review_ready,True,[restored_retrieval_source_not_official_text],"{'source_id_exists_in_corpus': True, 'citation...",Soru-cevap çifti corpus_final_strict_verified....


In [21]:
def evaluate_grounded_rag_system(model, system_name, eval_df):
    rows = []

    for i, row in eval_df.iterrows():
        question = row["question"]
        gold_answer = row["verified_answer"]

        retrieved_docs = final_retrieve(
            question,
            top_k=10
        )

        selected_context = select_answer_context(
            question,
            retrieved_docs,
            max_sentences=5
        )

        answer = generate_verified_answer(
            model,
            question,
            selected_context
        )

        rows.append({
            "system": system_name,
            "question": question,
            "gold_answer": gold_answer,
            "pred_answer": answer,
            "top_doc_id": retrieved_docs[0]["meta"]["doc_id"] if retrieved_docs else None,
            "top_title": retrieved_docs[0]["meta"]["title"] if retrieved_docs else None,
            "EM": exact_match(answer, gold_answer),
            "F1": token_f1(answer, gold_answer),
            "BLEU": bleu_score(answer, gold_answer),
            "ROUGE_L": rouge_l(answer, gold_answer),
            "Faithfulness": faithfulness_score(answer, selected_context),
            "Citation_Present": citation_present(answer),
            "Citation_Accuracy": citation_accuracy(answer, selected_context),
            "Hallucination_Rate": hallucination_rate(answer, selected_context)
        })

        if (i + 1) % 5 == 0:
            print(f"{system_name}: Processed {i+1}/{len(eval_df)}")

    return pd.DataFrame(rows)

In [27]:
base_grounded_eval_df = evaluate_grounded_rag_system(
    model=base_model,
    system_name="Base 7B + Final Grounded RAG Pipeline",
    eval_df=eval_df_final
)

base_grounded_metrics = base_grounded_eval_df.groupby("system")[
    [
        "EM",
        "F1",
        "BLEU",
        "ROUGE_L",
        "Faithfulness",
        "Citation_Present",
        "Citation_Accuracy",
        "Hallucination_Rate"
    ]
].mean().reset_index()

base_grounded_metrics

Base 7B + Final Grounded RAG Pipeline: Processed 5/100
Base 7B + Final Grounded RAG Pipeline: Processed 10/100
Base 7B + Final Grounded RAG Pipeline: Processed 15/100
Base 7B + Final Grounded RAG Pipeline: Processed 20/100
Base 7B + Final Grounded RAG Pipeline: Processed 25/100
Base 7B + Final Grounded RAG Pipeline: Processed 30/100
Base 7B + Final Grounded RAG Pipeline: Processed 35/100
Base 7B + Final Grounded RAG Pipeline: Processed 40/100
Base 7B + Final Grounded RAG Pipeline: Processed 45/100
Base 7B + Final Grounded RAG Pipeline: Processed 50/100
Base 7B + Final Grounded RAG Pipeline: Processed 55/100
Base 7B + Final Grounded RAG Pipeline: Processed 60/100
Base 7B + Final Grounded RAG Pipeline: Processed 65/100
Base 7B + Final Grounded RAG Pipeline: Processed 70/100
Base 7B + Final Grounded RAG Pipeline: Processed 75/100
Base 7B + Final Grounded RAG Pipeline: Processed 80/100
Base 7B + Final Grounded RAG Pipeline: Processed 85/100
Base 7B + Final Grounded RAG Pipeline: Processed 

,system,EM,F1,BLEU,ROUGE_L,Faithfulness,Citation_Present,Citation_Accuracy,Hallucination_Rate
0,Base 7B + Final Grounded RAG Pipeline,0.0,0.283414,0.073762,0.269182,0.984424,0.98,0.99,0.015576


In [28]:
ft_grounded_eval_df = evaluate_grounded_rag_system(
    model=ft_model,
    system_name="Fine-tuned 7B + Final Grounded RAG Pipeline",
    eval_df=eval_df_final
)

ft_grounded_metrics = ft_grounded_eval_df.groupby("system")[
    [
        "EM",
        "F1",
        "BLEU",
        "ROUGE_L",
        "Faithfulness",
        "Citation_Present",
        "Citation_Accuracy",
        "Hallucination_Rate"
    ]
].mean().reset_index()

ft_grounded_metrics

Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 5/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 10/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 15/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 20/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 25/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 30/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 35/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 40/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 45/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 50/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 55/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 60/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 65/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 70/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 75/100
Fine-tuned 7B + Final Grounded RAG Pipeline: Processed 80/100
Fine-tune

,system,EM,F1,BLEU,ROUGE_L,Faithfulness,Citation_Present,Citation_Accuracy,Hallucination_Rate
0,Fine-tuned 7B + Final Grounded RAG Pipeline,0.0,0.288301,0.078225,0.273104,0.993846,0.99,1.0,0.006154


In [29]:
old_results = pd.DataFrame([
    {
        "system": "Base 7B RAG + Old Retrieval",
        "EM": 0.000000,
        "F1": 0.098614,
        "BLEU": 0.003565,
        "ROUGE_L": 0.129709,
        "Faithfulness": 0.108574,
        "Citation_Present": None,
        "Citation_Accuracy": 0.033333,
        "Hallucination_Rate": 0.891426
    },
    {
        "system": "Fine-tuned 7B RAG + Old Retrieval",
        "EM": 0.000000,
        "F1": 0.126413,
        "BLEU": 0.009439,
        "ROUGE_L": 0.128675,
        "Faithfulness": 0.111441,
        "Citation_Present": None,
        "Citation_Accuracy": 0.033333,
        "Hallucination_Rate": 0.888559
    },
    {
        "system": "Base 7B RAG + Fine-tuned Legal Reranker",
        "EM": 0.000000,
        "F1": 0.093905,
        "BLEU": 0.005095,
        "ROUGE_L": 0.115153,
        "Faithfulness": 0.072390,
        "Citation_Present": None,
        "Citation_Accuracy": 0.000000,
        "Hallucination_Rate": 0.927610
    }
])

final_grounded_metrics = pd.concat(
    [
        base_grounded_metrics,
        ft_grounded_metrics
    ],
    ignore_index=True
)

final_comparison = pd.concat(
    [
        old_results,
        final_grounded_metrics
    ],
    ignore_index=True
)

final_comparison

/tmp/ipykernel_24237/1953963181.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_comparison = pd.concat(


,system,EM,F1,BLEU,ROUGE_L,Faithfulness,Citation_Present,Citation_Accuracy,Hallucination_Rate
0,Base 7B RAG + Old Retrieval,0.0,0.098614,0.003565,0.129709,0.108574,NaN,0.033333,0.891426
1,Fine-tuned 7B RAG + Old Retrieval,0.0,0.126413,0.009439,0.128675,0.111441,NaN,0.033333,0.888559
2,Base 7B RAG + Fine-tuned Legal Reranker,0.0,0.093905,0.005095,0.115153,0.072390,NaN,0.000000,0.927610
3,Base 7B + Final Grounded RAG Pipeline,0.0,0.283414,0.073762,0.269182,0.984424,0.98,0.990000,0.015576
4,Fine-tuned 7B + Final Grounded RAG Pipeline,0.0,0.288301,0.078225,0.273104,0.993846,0.99,1.000000,0.006154


In [30]:
final_eval_details = pd.concat(
    [
        base_grounded_eval_df,
        ft_grounded_eval_df
    ],
    ignore_index=True
)

final_eval_details.to_csv(
    "final_grounded_rag_eval_details_50.csv",
    index=False
)

final_comparison.to_csv(
    "final_grounded_rag_comparison_50.csv",
    index=False
)

print("Saved:")
print("final_grounded_rag_eval_details_50.csv")
print("final_grounded_rag_comparison_50.csv")

Saved:
final_grounded_rag_eval_details_50.csv
final_grounded_rag_comparison_50.csv
